# Mortalidad respiratoria/cardiovascular y rezago (lag) en la Ciudad de México

**Pregunta principal:** ¿Qué combinaciones de condiciones meteorológicas y niveles de contaminantes en la Ciudad de México, con qué rezago temporal (lag) respecto al evento, están más asociadas con incrementos en la mortalidad diaria por causas respiratorias y cardiovasculares, y qué patrones podrían usarse como señal temprana de días de mayor riesgo?

Este notebook es la primera evidencia de esta nueva etapa del proyecto (ver `docs/project_scope.md` y `docs/decisions.md`). Consume el panel diario ya construido y aplica el análisis exploratorio de correlación cruzada por rezago.

**Prerrequisitos** (ejecutar antes, desde la raíz del repositorio):

```bash
python -m src.processing.procesar_mortalidad_inegi data/raw/defunciones_2023.dbf data/raw/defunciones_2024.dbf
python -m src.processing.construir_panel_cdmx
python -m src.analysis.analisis_rezago
```

Este notebook asume que `data/CDMX_panel_diario.csv` ya existe. Las celdas de código son ejecutables, pero como el proyecto todavía no tiene los microdatos de mortalidad de INEGI descargados, no traen salidas precalculadas: se completan al correr el pipeline con datos reales.

## 1. Cargar el panel diario

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Permite importar los modulos de src/ (este notebook vive en notebooks/,
# un nivel por debajo de la raiz del repositorio).
sys.path.append(str(Path.cwd().parent))

from src.analysis.analisis_rezago import (
    DESENLACES,
    VARIABLES_BASE,
    calcular_correlaciones_por_rezago,
    mejor_rezago_por_variable,
)

RUTA_PANEL = "../data/CDMX_panel_diario.csv"

panel = pd.read_csv(RUTA_PANEL, parse_dates=["fecha"])
panel.head()

## 2. Diagnóstico inicial del panel

In [ ]:
print("Dimensiones:", panel.shape)
print("Rango de fechas:", panel["fecha"].min(), "a", panel["fecha"].max())
print("\nDuplicados por fecha:", panel.duplicated(subset=["fecha"]).sum())
print("\nValores faltantes en variables clave:")
panel[VARIABLES_BASE + DESENLACES].isna().sum()

In [ ]:
panel[DESENLACES].describe()

## 3. Correlación cruzada (CCF) por rezago

Para cada variable de exposición (clima y contaminantes) y cada desenlace de mortalidad, se calcula la correlación de Spearman entre la variable rezagada 0 a 14 días y el conteo de defunciones del día.

In [ ]:
tabla_correlaciones = calcular_correlaciones_por_rezago(panel)
tabla_correlaciones.head()

In [ ]:
resumen = mejor_rezago_por_variable(tabla_correlaciones)
resumen

## 4. Mapa de calor: correlación por variable y rezago

Un mapa de calor por desenlace ayuda a ver, de un vistazo, en qué rezago se concentra la asociación de cada variable (candidatos a señal temprana).

In [ ]:
def graficar_mapa_calor(tabla_correlaciones, desenlace):
    pivote = tabla_correlaciones[tabla_correlaciones["desenlace"] == desenlace].pivot(
        index="variable", columns="rezago_dias", values="correlacion_spearman"
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    limite = np.nanmax(np.abs(pivote.values)) if pivote.size else 1
    imagen = ax.imshow(pivote.values, cmap="RdBu_r", vmin=-limite, vmax=limite, aspect="auto")

    ax.set_xticks(range(len(pivote.columns)))
    ax.set_xticklabels(pivote.columns)
    ax.set_yticks(range(len(pivote.index)))
    ax.set_yticklabels(pivote.index)
    ax.set_xlabel("Rezago (días)")
    ax.set_title(f"Correlación de Spearman por rezago — {desenlace}")
    fig.colorbar(imagen, ax=ax, label="correlación")
    fig.tight_layout()
    return fig


for desenlace in DESENLACES:
    graficar_mapa_calor(tabla_correlaciones, desenlace)
    plt.show()

## 5. Interpretación (primera evidencia)

*Completar esta sección al ejecutar el notebook con los microdatos de mortalidad de INEGI ya descargados. Preguntas guía:*

- ¿Qué variable de exposición muestra la correlación más fuerte con la mortalidad respiratoria? ¿Y con la cardiovascular?
- ¿En qué rezago aparece esa correlación máxima? ¿Es consistente con la literatura de salud ambiental (efectos más rápidos para contaminantes gaseosos, más tardíos para partículas finas y temperaturas extremas)?
- ¿El rezago óptimo difiere entre mortalidad respiratoria y cardiovascular?
- ¿Hay señales (percentiles altos de exposición en el rezago identificado) que puedan usarse como umbral de alerta temprana?

## 6. Próximos pasos

- Ajustar un modelo de regresión Poisson/quasi-Poisson con términos rezagados y control por tendencia y estacionalidad (día de la semana, mes), en vez de solo correlación bivariada.
- Evaluar si sustituir/complementar Open-Meteo con mediciones de estación (SIMAT/RAMA-REDMET) cambia los rezagos identificados (ver `docs/decisions.md`).
- Extender la ventana temporal en cuanto se incorporen más años de microdatos de INEGI.